# GenAI-Net (RL4CRN) Tutorial 03 — Oscillator Discovery Task

Compact tutorial for the **oscillator** task.

Goal: learn a CRN that exhibits oscillatory dynamics with desired characteristics (often via a mean-level / frequency proxy reward).

---
## 0) Environment sanity check


In [ ]:
import os, sys, numpy as np

print("Python:", sys.version.split()[0])
print("CWD:", os.getcwd())


---
## 1) Imports

We use the standard interface layer:
- `Configurator`, `make_task`, `make_session_and_trainer`


In [ ]:
import numpy as np

from RL4CRN.utils.input_interface import (
    Configurator,
    make_task,
    make_session_and_trainer,
)


---
## 2) Notebook-local helpers


In [ ]:
def print_task_summary(task, max_preview=3):
    print("Task:", task.name)
    print("time_horizon:", task.time_horizon.shape, f"[0..{task.time_horizon[-1]}]")
    print("num scenarios:", len(task.u_list))
    if len(task.u_list) > 0:
        print(f"first {min(max_preview, len(task.u_list))} u:", task.u_list[:max_preview])
    print()

def run_smoke_reward(task, state, label=""):
    out = task.compute_reward(state)
    if isinstance(out, tuple):
        loss, info = out
    else:
        loss, info = out, {}
    print(f"[reward smoke{(' - ' + label) if label else ''}] loss={float(loss):.6g} | info_keys={list(info.keys())[:10]}")
    return out


---
## 3) Define the task (standalone)

Key knobs:
- `osc_w`: oscillator reward weights (task-specific)
- `t0`: burn-in time before evaluating oscillation properties


In [ ]:
species_labels = ["X_1","X_2","X_3"]

task = make_task(
    kind="oscillator",
    species_labels=species_labels,
    p=1,
    u_values=[1.0],
    ic=("constant", 0.01),
    t_f=50, n_t=120,
    osc_w=[0.4, 0.0, 0.2, 0.4],
    t0=20.0,
)

print_task_summary(task)


---
## 5) Full wiring + training loop (compact)

Pattern:
1. Configure `cfg`
2. Build `session, trainer`
3. Smoke-test reward on the template
4. Run a short training
5. Inspect the best


In [ ]:
cfg = Configurator.preset("fast")

# ---- Task ----
cfg.task.kind = "oscillator"
cfg.task.n_inputs = 1
cfg.task.p = 1
cfg.task.u_values = [1.0]
cfg.task.t_f = 50.0
cfg.task.N_t = 120
cfg.task.ic_value = 0.01
cfg.task.osc_w = [0.4, 0.0, 0.2, 0.4]
cfg.task.t0 = 20.0

# ---- Train ----
cfg.train.max_added_reactions = 4
cfg.train.epochs = 8
cfg.train.render_every = 2
cfg.train.seed = 2

session, trainer = make_session_and_trainer(cfg, device="auto")
print_task_summary(session.task)
run_smoke_reward(session.task, session.crn_template, label="template")

trainer.run(epochs=cfg.train.epochs, checkpoint_path=None)
trainer.inspect_best(plot=True)


---
## 6) Customize

Try:
- modifying `osc_w` to emphasize different oscillation properties
- increasing `t_f` and/or `N_t` if oscillations need longer to emerge
- changing ICs (`ic_value`) for stability
